# Phase 5 - Legal Statute Identification (LSI)
## Data cleaning, tokenization and analysis

Steps:
1. Config and load train / dev / test JSON
2. Clean text and labels
3. Integrity checks (ID overlap, duplicates, leakage)
4. Label analysis (frequency, rare statutes, dev-unseen, co-occurrence)
5. Length analysis and plots
6. Tokenization and chunking stats for long documents
7. Save outputs to /kaggle/working/phase5_lsi

In [ ]:
import os
import re
import json
import unicodedata
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd

DATA_DIR = '/kaggle/input/lsi-dataset/LSI'   # change to your dataset path
OUT_DIR = '/kaggle/working/phase5_lsi'
os.makedirs(OUT_DIR, exist_ok=True)

# Folder of the Phase-1 tokenizer. None = use a fallback word tokenizer.
PHASE1_TOKENIZER_DIR = None

MAX_LEN = 512       # tokens per chunk
STRIDE = 128        # overlap between chunks
RARE_THRESHOLD = 5  # statute seen <= this many times in train = rare

print('Data dir:', DATA_DIR)
print('Output dir:', OUT_DIR)

In [ ]:
def load_split(folder_name, file_name):
    path = os.path.join(DATA_DIR, folder_name, file_name)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(folder_name, file_name, '->', len(data), 'records')
    return data

train_raw = load_split('Train', 'train.json')
dev_raw = load_split('Train', 'dev.json')
test_raw = load_split('Test', 'test.json')

print('Keys in one train record:', list(train_raw[0].keys()))

In [ ]:
def clean_text(text):
    if text is None:
        return ''
    text = unicodedata.normalize('NFKC', str(text))
    kept = []
    for ch in text:
        if ch.isspace():
            kept.append(' ')
        elif unicodedata.category(ch) in ('Cc', 'Cs'):
            continue
        else:
            kept.append(ch)
    text = ''.join(kept)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_labels(labels):
    result = []
    for lab in labels:
        try:
            result.append(int(lab))
        except (TypeError, ValueError):
            continue
    return sorted(set(result))

def clean_labeled_split(records, name):
    seen_ids = set()
    rows = []
    stats = Counter()
    for rec in records:
        doc_id = str(rec.get('id', '')).strip()
        text = clean_text(rec.get('text', ''))
        if doc_id == '':
            stats['dropped_no_id'] += 1
            continue
        if doc_id in seen_ids:
            stats['dropped_duplicate_id'] += 1
            continue
        if text == '':
            stats['dropped_empty_text'] += 1
            continue
        labels = clean_labels(rec.get('labels', []))
        if len(labels) == 0:
            stats['dropped_no_labels'] += 1
            continue
        seen_ids.add(doc_id)
        rows.append({'id': doc_id, 'text': text, 'labels': labels})
    print(name, 'kept', len(rows), 'of', len(records))
    for key, value in stats.items():
        print('   ', key, value)
    return rows

In [ ]:
train_clean = clean_labeled_split(train_raw, 'train')
dev_clean = clean_labeled_split(dev_raw, 'dev')

# Test: keep every document, because the submission needs one prediction per test doc.
test_clean = []
for rec in test_raw:
    test_clean.append({'id': str(rec['id']), 'text': clean_text(rec.get('text', ''))})
print('test kept', len(test_clean), 'of', len(test_raw), '(all kept on purpose)')
print('test docs with empty text:', sum(1 for r in test_clean if r['text'] == ''))

In [ ]:
train_ids = set(r['id'] for r in train_clean)
dev_ids = set(r['id'] for r in dev_clean)
test_ids = set(r['id'] for r in test_clean)
print('ID overlap train-dev:', len(train_ids & dev_ids))
print('ID overlap train-test:', len(train_ids & test_ids))
print('ID overlap dev-test:', len(dev_ids & test_ids))

text_counts = Counter(r['text'] for r in train_clean)
dup_text_rows = sum(c - 1 for c in text_counts.values() if c > 1)
print('Duplicate document texts inside train:', dup_text_rows)

train_texts = set(text_counts.keys())
leaked = sum(1 for r in dev_clean if r['text'] in train_texts)
print('Dev docs whose text also appears in train:', leaked)

In [ ]:
train_label_counts = Counter()
for r in train_clean:
    train_label_counts.update(r['labels'])

dev_label_counts = Counter()
for r in dev_clean:
    dev_label_counts.update(r['labels'])

print('Distinct statutes in train:', len(train_label_counts))
print('Distinct statutes in dev:', len(dev_label_counts))

unseen_dev = [s for s in dev_label_counts if s not in train_label_counts]
print('Dev statutes never seen in train:', len(unseen_dev), unseen_dev[:20])

rare = [s for s, c in train_label_counts.items() if c <= RARE_THRESHOLD]
print('Rare statutes in train (count <=', RARE_THRESHOLD, '):', len(rare))

print('Most common statutes:', train_label_counts.most_common(10))
print('Least common statutes:', sorted(train_label_counts.items(), key=lambda x: x[1])[:10])

labels_per_doc = [len(r['labels']) for r in train_clean]
print('Labels per doc -> min', min(labels_per_doc), '| mean %.2f' % np.mean(labels_per_doc), '| max', max(labels_per_doc))
print('Docs with exactly one label:', sum(1 for n in labels_per_doc if n == 1))

pair_counts = Counter()
for r in train_clean:
    for a, b in combinations(r['labels'], 2):
        pair_counts[(a, b)] += 1
print('Top co-occurring statute pairs:', pair_counts.most_common(10))

In [ ]:
train_word_lens = [len(r['text'].split()) for r in train_clean]
dev_word_lens = [len(r['text'].split()) for r in dev_clean]
test_word_lens = [len(r['text'].split()) for r in test_clean]

len_summary = pd.DataFrame({
    'train': pd.Series(train_word_lens).describe(percentiles=[0.5, 0.9, 0.95, 0.99]),
    'dev': pd.Series(dev_word_lens).describe(percentiles=[0.5, 0.9, 0.95, 0.99]),
    'test': pd.Series(test_word_lens).describe(percentiles=[0.5, 0.9, 0.95, 0.99]),
})
print('Words per document:')
print(len_summary)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_word_lens, bins=50)
axes[0].set_title('Words per document (train)')
freqs = sorted(train_label_counts.values(), reverse=True)
axes[1].bar(range(len(freqs)), freqs)
axes[1].set_yscale('log')
axes[1].set_title('Statute frequency in train (log scale)')
plt.tight_layout()
plt.show()

In [ ]:
tokenizer = None
if PHASE1_TOKENIZER_DIR is not None:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(PHASE1_TOKENIZER_DIR)
    print('Loaded Phase-1 tokenizer:', type(tokenizer).__name__)
else:
    print('PHASE1_TOKENIZER_DIR is None -> using fallback word tokenizer')

WORD_RE = re.compile(r'\w+|[^\w\s]')

def fallback_tokenize(text):
    return WORD_RE.findall(text.lower())

vocab = {'<pad>': 0, '<unk>': 1}
if tokenizer is None:
    token_counter = Counter()
    for r in train_clean:
        token_counter.update(fallback_tokenize(r['text']))
    for tok, count in token_counter.items():
        if count >= 2:
            vocab[tok] = len(vocab)
    print('Fallback vocab size:', len(vocab))

def to_ids(text):
    if tokenizer is not None:
        return tokenizer(text, add_special_tokens=False)['input_ids']
    return [vocab.get(tok, 1) for tok in fallback_tokenize(text)]

In [ ]:
def chunk_ids(ids, max_len=MAX_LEN, stride=STRIDE):
    window = max_len
    step = max_len - stride
    chunks = []
    start = 0
    while True:
        chunks.append(ids[start:start + window])
        if start + window >= len(ids):
            break
        start += step
    return chunks

token_info = []
for split_name, split_data in [('train', train_clean), ('dev', dev_clean), ('test', test_clean)]:
    for r in split_data:
        ids = to_ids(r['text'])
        chunks = chunk_ids(ids)
        token_info.append({'split': split_name, 'id': r['id'], 'n_tokens': len(ids), 'n_chunks': len(chunks)})

token_df = pd.DataFrame(token_info)
print(token_df.groupby('split')[['n_tokens', 'n_chunks']].describe().T)

for split_name in ['train', 'dev', 'test']:
    sub = token_df[token_df['split'] == split_name]
    share_long = (sub['n_tokens'] > MAX_LEN).mean()
    print(split_name, '- share of docs longer than', MAX_LEN, 'tokens: %.1f%%' % (100 * share_long))

In [ ]:
def save_json(obj, file_name):
    path = os.path.join(OUT_DIR, file_name)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False)
    print('saved', path)

save_json(train_clean, 'train_clean.json')
save_json(dev_clean, 'dev_clean.json')
save_json(test_clean, 'test_clean.json')

label_df = pd.DataFrame({
    'statute_id': list(train_label_counts.keys()),
    'train_count': list(train_label_counts.values()),
})
label_df['dev_count'] = label_df['statute_id'].map(lambda s: dev_label_counts.get(s, 0))
label_df['is_rare'] = label_df['train_count'] <= RARE_THRESHOLD
label_df = label_df.sort_values('train_count', ascending=False)
label_df.to_csv(os.path.join(OUT_DIR, 'label_stats.csv'), index=False)
print('saved label_stats.csv')

token_df.to_csv(os.path.join(OUT_DIR, 'token_stats.csv'), index=False)
print('saved token_stats.csv')

if tokenizer is None:
    save_json(vocab, 'fallback_vocab.json')

print('Done.')